In [6]:
# loding the librarys
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns

In [5]:
# loding the dataset
df=pd.read_csv('P:\Internship_Final\Datasets\Amazon_cleaned_dataset.csv')

In [4]:
# making a copy
df1=df.copy()

##### first we will combine all the text features

In [7]:
# Create a combined text feature
df1["combined_text"] = (df1["name"].astype(str) + " " +df1["main_category"].astype(str) + " " +df1["sub_category"].astype(str))

- The recommendation system needs to understand the overall meaning of each product,
- not just individual columns. The product name, main category, and sub category each contain useful information,
- but when combined, they provide a more complete description of the product.

In [8]:
df1[["combined_text"]].head()

,combined_text
0,Lloyd 1.5 Ton 3 Star Inverter Split Ac (5 In 1...
1,LG 1.5 Ton 5 Star AI DUAL Inverter Split AC (C...
2,LG 1 Ton 4 Star Ai Dual Inverter Split Ac (Cop...
3,LG 1.5 Ton 3 Star AI DUAL Inverter Split AC (C...
4,Carrier 1.5 Ton 3 Star Inverter Split AC (Copp...


##### Now we will create the discount persentage

In [9]:
# Create discount percentage column
df1["discount_percentage"] = ((df1["actual_price"] - df1["discount_price"])/ df1["actual_price"]) * 100

- using only actual_price and discount_price doesn't tell us how much discount a customer is getting.
- thats why we created the discountedd persentage

#####  Now we will check the  Skewness of Numerical Features

In [11]:
# chcking the skewness
df1[["ratings","no_of_ratings","discount_price","actual_price","discount_percentage"]].skew()

c:\Users\mrinm\anaconda3\Lib\site-packages\pandas\core\nanops.py:1256: RuntimeWarning: invalid value encountered in subtract
  adjusted = values - mean


ratings                -1.637230
no_of_ratings          40.561040
discount_price         17.654710
actual_price           22.418793
discount_percentage          NaN
dtype: float64

##### Observation
- The no_of_ratings, discount_price, and actual_price features are highly positively skewed, indicating the presence of a few products with extremely high values.
- The ratings feature is slightly negatively skewed, which is expected because ratings are limited to a range of 1 to 5.
- These skewed distributions suggest that some numerical features may require transformation before being used in the recommendation model.

##### Now we will transfrom the skew numarical columns 

In [13]:
# now we will apply the log transformation.
df1["no_of_ratings"] = np.log1p(df1["no_of_ratings"])
df1["discount_price"] = np.log1p(df1["discount_price"])
df1["actual_price"] = np.log1p(df1["actual_price"])

In [17]:
# after the log transfrom
df1[["no_of_ratings","discount_price","actual_price"]].skew()

no_of_ratings    -0.153829
discount_price    0.858215
actual_price      0.349134
dtype: float64

##### Now we will convert the text feature into numaric feature

In [18]:
# importing  the library
from sentence_transformers import SentenceTransformer

In [19]:
# Now we will lode the pre train sentance tranfromar model
model = SentenceTransformer("all-MiniLM-L6-v2")

In [20]:
# Now we will make the text embeddinhg
embeddings = model.encode(df1["combined_text"].tolist(),batch_size=64,show_progress_bar=True,convert_to_numpy=True)

Batches:   0%|          | 0/8619 [00:00<?, ?it/s]

##### Now we will scale the numaric feature

In [21]:
# import the standard scaler
from sklearn.preprocessing import StandardScaler

In [30]:
# selecting the numaric feature 
numerical_features = ["ratings","no_of_ratings","discount_price","actual_price","discount_percentage"]

In [34]:
# This function checks whether every value is a valid finite number
np.isfinite(df1[numerical_features]).all()

ratings                 True
no_of_ratings           True
discount_price          True
actual_price            True
discount_percentage    False
dtype: bool

In [35]:
# we just reolacing the infinite values with NaN:
df1["discount_percentage"] = df1["discount_percentage"].replace([np.inf, -np.inf],np.nan)

In [36]:
# now we are creating the scaler and scaling the features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(df1[numerical_features])

##### Now we will combine the Text Embeddings and Numerical Features

In [37]:
# we are using this library so that we can horizontally combines multiple sparse matrices into a single sparse matrix.
from scipy.sparse import hstack

In [38]:

# Now we will combine text embeddings and scaled numerical features
hybrid_features = np.hstack((embeddings, scaled_features))

In [39]:
# Saving the feature-engineered dataset
df1.to_csv("../Datasets/Amazon_feature_dataset.csv", index=False)

##### Now we will save all the models 

In [40]:
import joblib

In [41]:
# first we will save the text embadding
joblib.dump(embeddings, "../Models/text_embeddings.pkl")

['../Models/text_embeddings.pkl']

In [42]:
# now we will save the scalled featured models
joblib.dump(scaled_features, "../Models/scaled_features.pkl")

['../Models/scaled_features.pkl']

In [43]:
# now we will save the scaler
joblib.dump(scaler, "../Models/scaler.pkl")

['../Models/scaler.pkl']

In [45]:
# now we will save the hybrid feature model
joblib.dump(hybrid_features, "../Models/hybrid_features.pkl")

['../Models/hybrid_features.pkl']